In [5]:
import pandas as pd
import duckdb
from pathlib import Path
import openpyxl
print("openpyxl is working") # for reading excel files

import matplotlib.pyplot as plt
import matplotlib.dates as mdates


openpyxl is working


In [6]:
df = pd.read_csv("../data/processed/netflix.csv")

# Register in DuckDB
duckdb.register("netflix",df)
df.shape


(458338, 15)

In [7]:
df.head()

,Unnamed: 0,country_name,country_iso2,week,country_category,country_weekly_rank,show_title,season_title,country_cumulative_weeks_in_top_10,global_category,global_weekly_rank,global_weekly_hours_viewed,runtime,global_weekly_views,global_cumulative_weeks_in_top_10
0,0,Argentina,AR,2026-03-15,Films,1,War Machine,NaN,2,Films (English),1.0,80600000.0,109.002,44400000.0,2.0
1,1,Argentina,AR,2026-03-15,Films,2,Strangers in the Park,NaN,2,Films (Non-English),10.0,1600000.0,115.998,800000.0,2.0
2,2,Argentina,AR,2026-03-15,Films,3,Joker: Folie à Deux,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN
3,3,Argentina,AR,2026-03-15,Films,4,Trolls Band Together,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN
4,4,Argentina,AR,2026-03-15,Films,5,Double Jeopardy,NaN,1,Films (English),5.0,6200000.0,106.002,3500000.0,1.0


In [8]:
duckdb.sql("""
DESCRIBE netflix
""").df()


,column_name,column_type,null,key,default,extra
0,Unnamed: 0,BIGINT,YES,None,None,None
1,country_name,VARCHAR,YES,None,None,None
2,country_iso2,VARCHAR,YES,None,None,None
3,week,VARCHAR,YES,None,None,None
4,country_category,VARCHAR,YES,None,None,None
5,country_weekly_rank,BIGINT,YES,None,None,None
6,show_title,VARCHAR,YES,None,None,None
7,season_title,VARCHAR,YES,None,None,None
8,country_cumulative_weeks_in_top_10,BIGINT,YES,None,None,None
9,global_category,VARCHAR,YES,None,None,None


## Dataset Checks
Validate structure first before analysis

In [9]:
duckdb.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT show_title) AS unique_show_titles,
    COUNT(DISTINCT week) AS unique_weeks,
    COUNT(DISTINCT country_name) AS unique_countries         
FROM netflix
""").df()


,total_rows,unique_show_titles,unique_weeks,unique_countries
0,458338,11060,246,94


Countries count by activity

In [10]:
duckdb.sql("""
    SELECT
    country_name,
    COUNT(*) AS appearances
FROM netflix
GROUP BY country_name
ORDER BY appearances DESC
""").df()


,country_name,appearances
0,Panama,4922
1,Dominican Republic,4922
2,Ecuador,4922
3,Argentina,4922
4,Venezuela,4922
...,...,...
89,Australia,4920
90,United States,4920
91,India,4920
92,South Africa,4920


Most Persistent Show Title

In [11]:
duckdb.sql("""
    SELECT
    country_name,
    COUNT(*) AS appearances
FROM netflix
GROUP BY country_name
ORDER BY appearances DESC
""").df()


,country_name,appearances
0,Venezuela,4922
1,El Salvador,4922
2,Honduras,4922
3,Uruguay,4922
4,Panama,4922
...,...,...
89,Ireland,4920
90,Hong Kong,4920
91,Kuwait,4920
92,Australia,4920


Best Performing Titles

In [12]:
duckdb.sql("""
    SELECT
    show_title,
    AVG(country_weekly_rank) AS avg_rank
FROM netflix
GROUP BY show_title
ORDER BY avg_rank ASC
LIMIT 20
""").df()


,show_title,avg_rank
0,The Call,1.000000
1,Our Secret Diary,1.000000
2,How To Choose the Right Husband?,1.000000
3,Boy,1.000000
4,Oi Vita Mia,1.000000
5,Andragogy,1.000000
6,Daddyku Gangster,1.000000
7,War Machine,1.107527
8,My Boss is a Serial Killer,1.500000
9,End of the Line,1.500000


### UX Team's Choice of Film & TV 
- UX has decided to only use 1 Swedish Film and 1 Swedish TV Series to use for their Dashboard
- Swedish TV Series = Young Royals
- Swedish Films = The Jönsson Gang Returns

In [13]:
duckdb.sql("""
    SELECT *
FROM netflix
WHERE show_title = 'Young Royals'
""").df()

,Unnamed: 0,country_name,country_iso2,week,country_category,country_weekly_rank,show_title,season_title,country_cumulative_weeks_in_top_10,global_category,global_weekly_rank,global_weekly_hours_viewed,runtime,global_weekly_views,global_cumulative_weeks_in_top_10
0,2100,Argentina,AR,2024-03-17,TV,9,Young Royals,Young Royals: Season 3,1,TV (Non-English),5.0,11500000.0,237.0,2900000.0,1.0
1,11922,Austria,AT,2024-03-24,TV,10,Young Royals,Young Royals: Season 3,2,TV (Non-English),9.0,7800000.0,294.0,1600000.0,2.0
2,11937,Austria,AT,2024-03-17,TV,5,Young Royals,Young Royals: Season 3,1,TV (Non-English),5.0,11500000.0,237.0,2900000.0,1.0
3,13360,Austria,AT,2022-11-06,TV,8,Young Royals,Young Royals: Season 2,1,TV (Non-English),3.0,18860000.0,NaN,NaN,1.0
4,31605,Belgium,BE,2024-03-24,TV,10,Young Royals,Young Royals: Season 3,2,TV (Non-English),9.0,7800000.0,294.0,1600000.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,425992,Ukraine,UA,2024-03-17,TV,9,Young Royals,Young Royals: Season 1,3,None,NaN,NaN,NaN,NaN,NaN
123,427409,Ukraine,UA,2022-11-06,TV,6,Young Royals,Young Royals: Season 2,1,TV (Non-English),3.0,18860000.0,NaN,NaN,1.0
124,428772,Ukraine,UA,2021-07-18,TV,9,Young Royals,Young Royals: Season 1,2,None,NaN,NaN,NaN,NaN,NaN
125,428787,Ukraine,UA,2021-07-11,TV,4,Young Royals,Young Royals: Season 1,1,TV (Non-English),8.0,9820000.0,NaN,NaN,1.0


In [ ]:
duckdb.sql("""
    SELECT *
FROM netflix
WHERE show_title = 'A Day and a Half'
""").df()

,Unnamed: 0,country_name,country_iso2,week,country_category,country_weekly_rank,show_title,season_title,country_cumulative_weeks_in_top_10,global_category,global_weekly_rank,global_weekly_hours_viewed,runtime,global_weekly_views,global_cumulative_weeks_in_top_10
0,14081,Austria,AT,2022-02-27,TV,9,Young Wallander,Young Wallander: Killer's Shadow,1,None,NaN,NaN,NaN,NaN,NaN
1,33764,Belgium,BE,2022-02-27,TV,9,Young Wallander,Young Wallander: Killer's Shadow,1,None,NaN,NaN,NaN,NaN,NaN
2,87895,Denmark,DK,2022-02-27,TV,5,Young Wallander,Young Wallander: Killer's Shadow,2,None,NaN,NaN,NaN,NaN,NaN
3,87915,Denmark,DK,2022-02-20,TV,5,Young Wallander,Young Wallander: Killer's Shadow,1,None,NaN,NaN,NaN,NaN,NaN
4,117423,Finland,FI,2022-02-27,TV,5,Young Wallander,Young Wallander: Killer's Shadow,2,None,NaN,NaN,NaN,NaN,NaN
5,117444,Finland,FI,2022-02-20,TV,6,Young Wallander,Young Wallander: Killer's Shadow,1,None,NaN,NaN,NaN,NaN,NaN
6,127269,Germany,DE,2022-02-27,TV,9,Young Wallander,Young Wallander: Killer's Shadow,1,None,NaN,NaN,NaN,NaN,NaN
7,161713,Iceland,IS,2022-02-27,TV,5,Young Wallander,Young Wallander: Killer's Shadow,2,None,NaN,NaN,NaN,NaN,NaN
8,161734,Iceland,IS,2022-02-20,TV,6,Young Wallander,Young Wallander: Killer's Shadow,1,None,NaN,NaN,NaN,NaN,NaN
9,230601,Luxembourg,LU,2022-02-27,TV,7,Young Wallander,Young Wallander: Killer's Shadow,1,None,NaN,NaN,NaN,NaN,NaN


# Netflix EDA | show title centric 

## Top Performers
What are the Top 10 titles per category per year?

In [15]:
duckdb.sql("""
WITH base AS (
    SELECT 
        DATE_TRUNC('year', STRPTIME(week, '%Y-%m-%d')) AS year,
        global_category,
        show_title,
        COUNT(*) AS appearances,
        AVG(country_weekly_rank) AS avg_rank,
        MAX(country_cumulative_weeks_in_top_10) AS longevity
    FROM netflix
    GROUP BY 
        year,
        global_category,
        show_title
),

ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY year, global_category
            ORDER BY 
                appearances DESC,        -- primary KPI
                avg_rank ASC             -- tie-breaker (lower is better)
        ) AS rank_in_category_year
    FROM base
)

SELECT *
FROM ranked
WHERE rank_in_category_year
ORDER BY year, global_category, rank_in_category_year
""").df()

,year,global_category,show_title,appearances,avg_rank,longevity,rank_in_category_year
0,2021-01-01,Films (English),Red Notice,581,3.399312,7,1
1,2021-01-01,Films (English),Army of Thieves,336,2.964286,4,2
2,2021-01-01,Films (English),Love Hard,296,4.195946,5,3
3,2021-01-01,Films (English),Kate,287,3.233449,4,4
4,2021-01-01,Films (English),The Guilty,285,2.343860,4,5
...,...,...,...,...,...,...,...
19357,2026-01-01,None,Hometown Cha-Cha-Cha,1,10.000000,18,987
19358,2026-01-01,None,Cashero,1,10.000000,5,988
19359,2026-01-01,None,Exit Wounds,1,10.000000,3,989
19360,2026-01-01,None,Bard of Blood,1,10.000000,2,990
